# U.S. Medical Insurance Costs--Step2. Hypothesis Testing

In [1]:
import pandas as pd

In [2]:
ins_cleaned_df = pd.read_csv("/Users/shenkong/Desktop/Projects/pj02_US_Medical_Insurance_Costs/data/ins_cleaned.csv")

In [3]:
# 1. Continuous Variables: age/bmi/children vs charges
# Use Correlation + p-value
from scipy.stats import pearsonr
corr_age, p_age = pearsonr(ins_cleaned_df["age"], ins_cleaned_df["charges"])
print(corr_age, p_age)
# age and charges show a statistically significant positive correlation, since corr > 0 and p < 0.05

0.2983082125097864 6.975748762690151e-29


In [4]:
corr_bmi, p_bmi = pearsonr(ins_cleaned_df["bmi"], ins_cleaned_df["charges"])
print(corr_bmi, p_bmi)
# bmi and charges show a statistically significant positive correlation, since corr > 0 and p < 0.05

0.19840083122624935 2.468040426451341e-13


In [5]:
corr_children, p_children = pearsonr(ins_cleaned_df["children"], ins_cleaned_df["charges"])
print(corr_children, p_children)
# children and charges show a statistically significant positive correlation, since corr > 0 and p < 0.05

0.06738935083963249 0.013717026292987156


In [6]:
# 2. Categorical Variables: smoker/sex/region vs charges
# Use t-test(Yes or No) / ANOVA(Analysis of Variance)
from scipy.stats import ttest_ind
yes = ins_cleaned_df[ins_cleaned_df["smoker"] == "yes"]["charges"]
no = ins_cleaned_df[ins_cleaned_df["smoker"] == "no"]["charges"]

t_stat, p_smoker = ttest_ind(yes, no)
print(t_stat, p_smoker)
# There is a statistically significant difference in medical charges between smokers and non-smokers since p < 0.05, and smokers tend to have significantly higher charges since t_stat > 0 (mean(yes) - mean(no) > 0)

46.64479459840305 1.4067220949376494e-282


In [7]:
male = ins_cleaned_df[ins_cleaned_df["sex"] == "male"]["charges"]
female = ins_cleaned_df[ins_cleaned_df["sex"] == "female"]["charges"]

t_stat, p_sex = ttest_ind(male, female)
print(t_stat, p_sex)
# There is a statistically significant difference in medical charges between men and women since p < 0.05, and men tend to have significantly higher charges since t_stat > 0 (mean(male) - mean(female) > 0)

2.124391307062026 0.033820791995119504


In [8]:
ins_cleaned_df["region"].unique()
# Use ANOVA instead of t-test, there are more than 2 values in this column

<StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str

In [9]:
# ANOVA tests whether there are statistically significant differences among all groups at once
from scipy.stats import f_oneway

four_regions = [
    ins_cleaned_df[ins_cleaned_df["region"] == r]["charges"]
    for r in ins_cleaned_df["region"].unique()
]

f_stat, p_region = f_oneway(*four_regions)
print(f_stat, p_region)
# There is a statistically significant difference in medical charges across different regions since F=2.93, p=0.033, suggesting that at least one region has a different mean insurance charge compared to others

2.926139903662776 0.03276288025447234


In [10]:
# Summary table:
summary_table = pd.DataFrame({
                            "Variable": [
                                "age",
                                "bmi",
                                "children",
                                "smoker",
                                "sex",
                                "region",
                            ],
                            "Test":[
                                "Pearson",
                                "Pearson",
                                "Pearson",
                                "t-test",
                                "t-test",
                                "ANOVA"
                            ],
                            "p-value":[
                                p_age,
                                p_bmi,
                                p_children,
                                p_smoker,
                                p_sex,
                                p_region
                            ]
})
summary_table

,Variable,Test,p-value
0,age,Pearson,6.975749e-29
1,bmi,Pearson,2.468040e-13
2,children,Pearson,1.371703e-02
3,smoker,t-test,1.406722e-282
4,sex,t-test,3.382079e-02
5,region,ANOVA,3.276288e-02


In [11]:
# Sig_label Function:
def significance_label(p):
    if p < 0.001:
        return "Yes (Strong)"
    elif p < 0.05:
        return "Yes"
    else:
        return "No"

In [12]:
summary_table["Significant?"] = (
    summary_table["p-value"]
    .apply(significance_label)
)

In [13]:
summary_table["p-value"] = (
    summary_table["p-value"]
    .astype(float)  # Convert string to float datatype
    .apply(lambda x: round(x, 3))  # Round the floats to 3 decimal places
)
summary_table

,Variable,Test,p-value,Significant?
0,age,Pearson,0.000,Yes (Strong)
1,bmi,Pearson,0.000,Yes (Strong)
2,children,Pearson,0.014,Yes
3,smoker,t-test,0.000,Yes (Strong)
4,sex,t-test,0.034,Yes
5,region,ANOVA,0.033,Yes


In [14]:
ins_cleaned_df.to_csv('/Users/shenkong/Desktop/Projects/pj02_US_Medical_Insurance_Costs/ins_cleaned.csv', index=False, encoding='utf-8')  # Output the cleaned data